In [9]:
import CalculatedFieldSubroutines as cfs

#

import numpy as np

import pandas as pd

import math

#

import os

#

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation

from matplotlib.animation import FFMpegWriter

#

import cartopy.crs as ccrs

from cartopy.io.img_tiles import GoogleTiles

#

from pandasgui import show

#

import warnings

In [10]:
warnings.filterwarnings( 'ignore' )

In [11]:
def display_poi( df, poi_index, poi_relative_time_interval, poi_label, save_path, animation = False ):

    cfs.Acceleration( df, time_interval = 0.25 )

    #

    time_array = np.array( df[ 'time' ] )

    #

    interval_start, interval_end = poi_relative_time_interval

    poi_relative_time_interval_indexes = cfs.single_moving_window_indexer( moving_col_array = time_array,
                                                                           index = poi_index,
                                                                           relative_moving_window_interval = ( interval_start, interval_end ) )

    #

    time_array = time_array - time_array[ poi_index ]

    time_array = time_array[ poi_relative_time_interval_indexes ]

    time_array = 1e-9 * time_array

    #

    poi_index = int( np.where( time_array == 0 )[ 0 ][ 0 ] )

    #

    poi_progress_value = np.array( df[ 'ProgressAlongRoute' ] )[ poi_relative_time_interval_indexes ][ poi_index ]

    #

    longitude_array = np.array( df[ 'longitude' ] )[ poi_relative_time_interval_indexes ]

    latitude_array = np.array( df[ 'latitude' ] )[ poi_relative_time_interval_indexes ]

    speed_array = np.array( df[ 'speedMps' ] )[ poi_relative_time_interval_indexes ]

    acceleration_array = np.array( df[ 'Acceleration' ] )[ poi_relative_time_interval_indexes ]

    steering_array = np.array( df[ 'steeringPercentage' ] )[ poi_relative_time_interval_indexes ]

    throttle_array = np.array( df[ 'throttlePercentage' ] )[ poi_relative_time_interval_indexes ]

    brake_array = np.array( df[ 'brakePercentage' ] )[ poi_relative_time_interval_indexes ]

    latlonstddev_array = np.array( df[ 'LatLonTotalStdDev' ] )[ poi_relative_time_interval_indexes ]

    #

    driving_mode_array = np.array( df[ 'BinaryDrivingMode' ] )[ poi_relative_time_interval_indexes ]

    #

    manual_indexes = list( np.where( driving_mode_array == 0 )[ 0 ] )

    auto_indexes = list( np.where( driving_mode_array == 1 )[ 0 ] )

    #

    manual_indexes = [ index for index in manual_indexes if int( index ) != poi_index ]

    auto_indexes = [ index for index in auto_indexes if int( index ) != poi_index ]

    ###

    fig = plt.figure( constrained_layout = True, figsize = ( 15, 7.5 ) )

    #

    ax = fig.add_gridspec( 3, 6 )

    ##

    street_map = GoogleTiles( style = 'street' )

    min_Lot, max_Lot, min_Lat, max_Lat = np.min( longitude_array ), np.max( longitude_array ), np.min( latitude_array ), np.max( latitude_array )

    expansion_coeff = 0.1

    Lot_adjustment, Lat_adjustment = expansion_coeff * np.abs( min_Lot - max_Lot ), expansion_coeff * np.abs( min_Lat - max_Lat )

    min_Lot, max_Lot, min_Lat, max_Lat = min_Lot - Lot_adjustment, max_Lot + Lot_adjustment, min_Lat - Lat_adjustment, max_Lat + Lat_adjustment

    #

    ax_map = fig.add_subplot( ax[ 0 : 3, 0 : 2 ], projection = street_map.crs )

    ax_map.set_title( 'Detailed Map' )

    ax_map.set_extent( [ min_Lot , max_Lot, min_Lat, max_Lat ], ccrs.PlateCarree() )

    ax_map.add_image( street_map, 20 )

    ##

    route_color = cfs.give_route( df[ 'groupMetadataID' ][ 0 ] )

    if ( route_color == 'Red' ):

        route_minLat, route_maxLat, route_minLon, route_maxLon = 39.3124906671225, 39.40448581121833, -82.10582075491102, -81.9788697421841

        route_reference_gmID = '9798fe24-f143-11ee-ba78-fb353e7798cd'

    elif ( route_color == 'Green' ):

        route_minLat, route_maxLat, route_minLon, route_maxLon = 39.28843222457655, 39.33215131664078, -82.46754600607878, -82.4289058208249

        route_reference_gmID = '3a7dc9a6-f042-11ee-b974-fb353e7798cd'

    elif ( route_color == 'Blue' ):

        route_minLat, route_maxLat, route_minLon, route_maxLon = 39.31899122387631, 39.38797086107572, -82.14449476722909, -82.10122278485707

        route_reference_gmID = '3d2a80f0-ec81-11ee-b297-3b0ad9d5d6c6'

    route_Lot_adjustment, route_Lat_adjustment = expansion_coeff * np.abs( route_minLon - route_maxLon ), expansion_coeff * np.abs( route_minLat - route_maxLat )

    route_minLon, route_maxLon, route_minLat, route_maxLat = route_minLon - route_Lot_adjustment, route_maxLon + route_Lot_adjustment, \
                                                             route_minLat - route_Lat_adjustment, route_maxLat + route_Lat_adjustment

    #

    ax_route_map = fig.add_subplot( ax[ 0 : 3, 4 : 6 ], projection = street_map.crs )

    ax_route_map.set_title( f'Location on Route' )

    ax_route_map.set_extent( [ route_minLon , route_maxLon, route_minLat, route_maxLat ], ccrs.PlateCarree() )

    ax_route_map.add_image( street_map, 14 )

    ##

    min_time, max_time = np.min( time_array ), np.max( time_array )

    time_adjustment = 0.5 * expansion_coeff * np.abs( min_time - max_time )

    min_time, max_time = min_time - time_adjustment, max_time + time_adjustment

    ##

    min_speed, max_speed = np.min( speed_array ), np.max( speed_array )

    speed_adjustment = expansion_coeff * np.abs( min_speed - max_speed )

    min_speed, max_speed = min_speed - speed_adjustment, max_speed + speed_adjustment

    #

    ax_speed = fig.add_subplot( ax[ 0, 2 ] )

    ax_speed.set_title( 'Speed [m/s]' )

    ax_speed.set_xlim( min_time, max_time )

    ax_speed.set_ylim( min_speed, max_speed )

    ax_speed.axhline( y = 0, color = 'black', ls = '--' )

    ##

    min_acceleration, max_acceleration = np.min( acceleration_array ), np.max( acceleration_array )

    acceleration_adjustment = expansion_coeff * np.abs( min_acceleration - max_acceleration )

    min_acceleration, max_acceleration = min_acceleration - acceleration_adjustment, max_acceleration + acceleration_adjustment

    #

    ax_acceleration = fig.add_subplot( ax[ 0, 3 ] )

    ax_acceleration.set_title( 'Acceleration [m/s^2]' )

    ax_acceleration.set_xlim( min_time, max_time )

    ax_acceleration.set_ylim( min_acceleration, max_acceleration )

    ax_acceleration.axhline( y = 0, color = 'black', ls = '--' )

    ##

    min_steering, max_steering = np.min( steering_array ), np.max( steering_array )

    steering_adjustment = expansion_coeff * np.abs( min_steering - max_steering )

    min_steering, max_steering = min_steering - steering_adjustment, max_steering + steering_adjustment

    #

    ax_steering = fig.add_subplot( ax[ 2, 2 ] )

    ax_steering.set_title( 'Steering [%]' )

    ax_steering.set_xlim( min_time, max_time )

    ax_steering.set_ylim( min_steering, max_steering )

    ax_steering.axhline( y = 0, color = 'black', ls = '--' )

    ##

    min_throttle, max_throttle = np.min( throttle_array ), np.max( throttle_array )

    throttle_adjustment = expansion_coeff * np.abs( min_throttle - max_throttle )

    min_throttle, max_throttle = min_throttle - throttle_adjustment, max_throttle + throttle_adjustment

    #

    ax_throttle = fig.add_subplot( ax[ 1, 2 ] )

    ax_throttle.set_title( 'Throttle [%]' )

    ax_throttle.set_xlim( min_time, max_time )

    ax_throttle.set_ylim( min_throttle, max_throttle )

    ax_throttle.axhline( y = 0, color = 'black', ls = '--' )

    ##

    min_brake, max_brake = np.min( brake_array ), np.max( brake_array )

    brake_adjustment = expansion_coeff * np.abs( min_brake - max_brake )

    min_brake, max_brake = min_brake - brake_adjustment, max_brake + brake_adjustment

    #

    ax_brake = fig.add_subplot( ax[ 1, 3 ] )

    ax_brake.set_title( 'Brake [%]' )

    ax_brake.set_xlim( min_time, max_time )

    ax_brake.set_ylim( min_brake, max_brake )

    ax_brake.axhline( y = 0, color = 'black', ls = '--' )

    ##

    min_latlonstddev, max_latlonstddev = np.min( latlonstddev_array ), np.max( latlonstddev_array )

    latlonstddev_adjustment = expansion_coeff * np.abs( min_latlonstddev - max_latlonstddev )

    min_latlonstddev, max_latlonstddev = min_latlonstddev - latlonstddev_adjustment, max_latlonstddev + latlonstddev_adjustment

    #

    ax_latlonstddev = fig.add_subplot( ax[ 2, 3 ] )

    ax_latlonstddev.set_title( 'Positional StdDev [m]' )

    ax_latlonstddev.set_xlim( min_time, max_time )

    ax_latlonstddev.set_ylim( min_latlonstddev, max_latlonstddev )

    ax_latlonstddev.axhline( y = 0, color = 'black', ls = '--' )

    ##

    fig.supxlabel( 'Time [sec]', x = 0.5, fontsize = 15 )

    fig.suptitle( f'POI: { poi_label } on { route_color } Route', fontsize = 20 )

    ###

    reference_color = route_color.lower()

    reference_style = { 'marker': '.', 'ls': '', 'color': f'{ reference_color }', 'ms': 5 }

    poi_style = { 'marker': 'D', 'ls': '', 'color': 'black', 'ms': 5 }

    manual_style = { 'marker': 'o', 'ls': '', 'color': 'tab:orange', 'ms': 2 }

    auto_style = { 'marker': 'o', 'ls': '', 'color': 'tab:blue', 'ms': 2 }

    manual_style2 = { 'marker': '.', 'ls': '', 'color': 'tab:orange', 'ms': 6 }

    auto_style2 = { 'marker': '.', 'ls': '', 'color': 'tab:blue', 'ms': 6 }

    start_style = { 'marker': 'x', 'ls': '', 'color': 'black', 'ms': 6 }

    ##

    route_reference_gmID_df = cfs.retrieve_gmID_preprocessed_moving_data_v3_reduced( gmID = route_reference_gmID, moving_window = 0 )

    ax_route_map.plot( route_reference_gmID_df[ 'longitude' ], route_reference_gmID_df[ 'latitude' ], transform = ccrs.PlateCarree(), **reference_style )

    #

    ax_route_map.plot( [ longitude_array[ poi_index ] ], [ latitude_array[ poi_index ] ], transform = ccrs.PlateCarree(), 
                       label = f'Route Progress: { poi_progress_value :.3f}', **poi_style )

    ax_route_map.legend()

    ##

    partial_ax_list = [ ax_speed, ax_steering, ax_throttle, ax_brake, ax_latlonstddev, ax_acceleration ]

    array_list = [ speed_array, steering_array, throttle_array, brake_array, latlonstddev_array, acceleration_array ]

    ##

    if ( animation == False ):

        ax_map.plot( longitude_array[ manual_indexes ], latitude_array[ manual_indexes ], transform = ccrs.PlateCarree(), label = 'Manual Driving Mode', **manual_style2 )

        ax_map.plot( longitude_array[ auto_indexes ], latitude_array[ auto_indexes ], transform = ccrs.PlateCarree(), label = 'Automatic Driving Mode', **auto_style2 )

        ax_map.plot( [ longitude_array[ poi_index ] ], [ latitude_array[ poi_index ] ], transform = ccrs.PlateCarree(), label = f'POI', **poi_style )

        ax_map.plot( [ longitude_array[ 0 ] ], [ latitude_array[ 0 ] ], transform = ccrs.PlateCarree(), label = 'Starting Point', **start_style )

        ax_map.legend()

        ##

        for current_ax, array in zip( partial_ax_list, array_list ):

            current_ax.plot( time_array[ manual_indexes ], array[ manual_indexes ], **manual_style )

            current_ax.plot( time_array[ auto_indexes ], array[ auto_indexes ], **auto_style )

            current_ax.plot( [ time_array[ poi_index ] ], [ array[ poi_index ] ], **poi_style )

        ###

        os.chdir( save_path )

        plt.savefig( f'{ poi_label }.png' )

        plt.close()

    elif ( animation == True ):

        animated_ax_map_manual, = ax_map.plot( [], [], transform = ccrs.PlateCarree(), label = 'Manual Driving Mode', **manual_style2 )

        animated_ax_map_auto, = ax_map.plot( [], [], transform = ccrs.PlateCarree(), label = 'Automatic Driving Mode', **auto_style2 )

        animated_ax_map_poi, = ax_map.plot( [], [], transform = ccrs.PlateCarree(), label = f'POI', **poi_style )

        ax_map.legend()

        ##

        animated_speed_ax_manual, = ax_speed.plot( [], [], **manual_style )

        animated_speed_ax_auto, = ax_speed.plot( [], [], **auto_style )

        animated_speed_ax_poi, = ax_speed.plot( [], [], **poi_style )

        #

        animated_steering_ax_manual, = ax_steering.plot( [], [], **manual_style )

        animated_steering_ax_auto, = ax_steering.plot( [], [], **auto_style )

        animated_steering_ax_poi, = ax_steering.plot( [], [], **poi_style )

        #

        animated_throttle_ax_manual, = ax_throttle.plot( [], [], **manual_style )

        animated_throttle_ax_auto, = ax_throttle.plot( [], [], **auto_style )

        animated_throttle_ax_poi, = ax_throttle.plot( [], [], **poi_style )

        #

        animated_brake_ax_manual, = ax_brake.plot( [], [], **manual_style )

        animated_brake_ax_auto, = ax_brake.plot( [], [], **auto_style )

        animated_brake_ax_poi, = ax_brake.plot( [], [], **poi_style )

        #

        animated_latlonstddev_ax_manual, = ax_latlonstddev.plot( [], [], **manual_style )

        animated_latlonstddev_ax_auto, = ax_latlonstddev.plot( [], [], **auto_style )

        animated_latlonstddev_ax_poi, = ax_latlonstddev.plot( [], [], **poi_style )

        #

        animated_acceleration_ax_manual, = ax_acceleration.plot( [], [], **manual_style )

        animated_acceleration_ax_auto, = ax_acceleration.plot( [], [], **auto_style )

        animated_acceleration_ax_poi, = ax_acceleration.plot( [], [], **poi_style )

        ##

        def update( frame_num ):

            temp_driving_mode_array = driving_mode_array[ : frame_num + 1 ]

            #

            temp_manual_indexes = list( np.where( temp_driving_mode_array == 0 )[ 0 ] )

            temp_auto_indexes = list( np.where( temp_driving_mode_array == 1 )[ 0 ] )

            if ( ( poi_index in temp_manual_indexes ) or ( poi_index in temp_auto_indexes ) ):

                include_poi = True

            else:

                include_poi = False

            #

            temp_manual_indexes = [ index for index in temp_manual_indexes if int( index ) != poi_index ]

            temp_auto_indexes = [ index for index in temp_auto_indexes if int( index ) != poi_index ]

            ##

            animated_ax_map_manual.set_data( longitude_array[ temp_manual_indexes ], latitude_array[ temp_manual_indexes ] )

            animated_ax_map_auto.set_data( longitude_array[ temp_auto_indexes ], latitude_array[ temp_auto_indexes ] )

            if ( include_poi == True ):

                animated_ax_map_poi.set_data( [ longitude_array[ poi_index ] ], [ latitude_array[ poi_index ] ] )

            else:

                animated_ax_map_poi.set_data( [], [] )

            #

            animated_speed_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 0 ][ temp_manual_indexes ] )

            animated_speed_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 0 ][ temp_auto_indexes ] )

            #

            animated_steering_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 1 ][ temp_manual_indexes ] )

            animated_steering_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 1 ][ temp_auto_indexes ] )

            #

            animated_throttle_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 2 ][ temp_manual_indexes ] )

            animated_throttle_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 2 ][ temp_auto_indexes ] )

            #

            animated_brake_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 3 ][ temp_manual_indexes ] )

            animated_brake_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 3 ][ temp_auto_indexes ] )

            #

            animated_latlonstddev_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 4 ][ temp_manual_indexes ] )

            animated_latlonstddev_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 4 ][ temp_auto_indexes ] )

            #

            animated_acceleration_ax_manual.set_data( time_array[ temp_manual_indexes ], array_list[ 5 ][ temp_manual_indexes ] )

            animated_acceleration_ax_auto.set_data( time_array[ temp_auto_indexes ], array_list[ 5 ][ temp_auto_indexes ] )

            #

            if ( include_poi == True ):

                animated_speed_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 0 ][ poi_index ] ] )

                animated_steering_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 1 ][ poi_index ] ] )

                animated_throttle_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 2 ][ poi_index ] ] )

                animated_brake_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 3 ][ poi_index ] ] )

                animated_latlonstddev_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 4 ][ poi_index ] ] )

                animated_acceleration_ax_poi.set_data( [ time_array[ poi_index ] ], [ array_list[ 5 ][ poi_index ] ] )

            else:

                animated_speed_ax_poi.set_data( [], [] )

                animated_steering_ax_poi.set_data( [], [] )

                animated_throttle_ax_poi.set_data( [], [] )

                animated_brake_ax_poi.set_data( [], [] )

                animated_latlonstddev_ax_poi.set_data( [], [] )

                animated_acceleration_ax_poi.set_data( [], [] )

        ##

        num_of_frames = len( driving_mode_array )

        #

        seconds_per_frame = 0.03168499496201086

        #

        animation = FuncAnimation( fig = fig, func = update, frames = num_of_frames )

        ###

        animation.save( f'{ save_path }{ poi_label }.mp4', writer = FFMpegWriter( fps = 1 / seconds_per_frame ) )

        plt.close()

In [12]:
gmID_list = cfs.list_whitelisted_gmIDs()

In [13]:
gmID_list = ['25e27b86-f06a-11ee-b9a3-fb353e7798cd',
 'ba87f3ec-f07e-11ee-b9b4-fb353e7798cd',
 '7cbd932e-f244-11ee-bb3f-fb353e7798cd',
 '6af236d6-d98f-11ee-a158-97f8443fd730',
 '1ee938a2-f172-11ee-baa6-fb353e7798cd',
 'baf0e4be-bede-11ee-835b-599066b5eb60',
 '3343fd3c-eb87-11ee-b297-3b0ad9d5d6c6',
 '5c7a9ab2-f13b-11ee-ba72-fb353e7798cd',
 'fe973c9c-f53c-11ee-8afa-cb629b0d53e6',
 '57d240d6-ea4d-11ee-b297-3b0ad9d5d6c6',
 'f8fd0fd8-f243-11ee-bb3f-fb353e7798cd',
 '06cbdbc0-db4d-11ee-a158-97f8443fd730',
 '25d3bdc8-ecbc-11ee-b297-3b0ad9d5d6c6',
 '2bc6ebb8-a529-11ee-88ec-eb6a8d5269b4',
 '25135418-f250-11ee-bb4a-fb353e7798cd',
 'bf9157f0-f16b-11ee-ba9e-fb353e7798cd',
 '43914d48-ed85-11ee-9385-ef789ffde1d3',
 '5a4bccf4-effe-11ee-b966-fb353e7798cd',
 'c0624e24-d9aa-11ee-a158-97f8443fd730',
 '513a670c-eea9-11ee-9385-ef789ffde1d3',
 '2d35c522-eba2-11ee-b297-3b0ad9d5d6c6',
 'b31aca98-cb95-11ee-909c-e1dc60cf66f9',
 'd7cb9c92-f164-11ee-ba97-fb353e7798cd',
 '3ce8a358-edd8-11ee-9385-ef789ffde1d3',
 '271fee10-cb8b-11ee-909c-e1dc60cf66f9',
 'cb205756-ec43-11ee-b297-3b0ad9d5d6c6',
 'e8a8b2be-edbf-11ee-9385-ef789ffde1d3',
 'fc119dfc-eb67-11ee-b297-3b0ad9d5d6c6',
 'c14299be-f180-11ee-bab0-fb353e7798cd',
 'd94ef300-ed60-11ee-9385-ef789ffde1d3',
 'df6c3fb4-f200-11ee-bb07-fb353e7798cd',
 '1bbbfbae-c839-11ee-a7fc-dd032dba19e8',
 '3a7dc9a6-f042-11ee-b974-fb353e7798cd',
 '3ea96640-ea37-11ee-b297-3b0ad9d5d6c6',
 '072ef896-cbac-11ee-909c-e1dc60cf66f9',
 '211bdb36-f0da-11ee-ba1b-fb353e7798cd',
 'e6d7d384-db40-11ee-a158-97f8443fd730',
 '0f4f0a06-ea98-11ee-b297-3b0ad9d5d6c6',
 '9830d896-d2dc-11ee-b437-336917683bb8',
 '171c50bc-f106-11ee-ba42-fb353e7798cd',
 '88180f82-ed4f-11ee-9385-ef789ffde1d3',
 '8347b862-efad-11ee-b966-fb353e7798cd',
 'd62ee6e8-ed02-11ee-9385-ef789ffde1d3',
 'f41cbd44-eff8-11ee-b966-fb353e7798cd',
 '8adb6498-f04d-11ee-b981-fb353e7798cd',
 'cbdc93f4-f255-11ee-bb4e-fb353e7798cd',
 'fe0395f0-f1ea-11ee-baf9-fb353e7798cd',
 '40706f50-f03b-11ee-b96e-fb353e7798cd',
 '90101c36-a621-11ee-88ec-eb6a8d5269b4',
 '559495ca-d270-11ee-b437-336917683bb8',
 '326699c2-ecd8-11ee-b297-3b0ad9d5d6c6',
 '7e3d64da-f12d-11ee-ba68-fb353e7798cd',
 '14b6bc9c-f064-11ee-b998-fb353e7798cd',
 '236836f6-f1dd-11ee-bae8-fb353e7798cd',
 '457dc5ee-f02a-11ee-b966-fb353e7798cd',
 '848e44a6-f134-11ee-ba6d-fb353e7798cd',
 'd3698592-ef9d-11ee-b966-fb353e7798cd',
 '5976b77a-a504-11ee-88ec-eb6a8d5269b4',
 'a17c1280-ea10-11ee-b297-3b0ad9d5d6c6',
 '5f7ce340-f1c8-11ee-bae0-fb353e7798cd',
 '65cfbfd6-f396-11ee-bb4e-fb353e7798cd',
 'fc211bb2-efca-11ee-b966-fb353e7798cd',
 'f43b6a70-f01e-11ee-b966-fb353e7798cd',
 '5fcc4fd8-ea71-11ee-b297-3b0ad9d5d6c6',
 '94c53148-eeed-11ee-9385-ef789ffde1d3',
 '35518ec4-f153-11ee-ba88-fb353e7798cd',
 'e7b934a8-ef1a-11ee-9385-ef789ffde1d3',
 '88b0613a-d35d-11ee-b437-336917683bb8',
 'e9d67bf2-ec35-11ee-b297-3b0ad9d5d6c6',
 '20f0b890-ec64-11ee-b297-3b0ad9d5d6c6',
 '9189a2a8-f121-11ee-ba5b-fb353e7798cd',
 '622bd2e8-f0e4-11ee-ba1f-fb353e7798cd',
 '20cbfe8c-ea2b-11ee-b297-3b0ad9d5d6c6',
 '3151e9e2-eff3-11ee-b966-fb353e7798cd',
 'cf6fdf3a-eaa3-11ee-b297-3b0ad9d5d6c6',
 'aa86a660-dc05-11ee-a158-97f8443fd730',
 '69ab88ec-dc17-11ee-a158-97f8443fd730',
 '787d9684-d2c2-11ee-b437-336917683bb8',
 '61b12e7a-f234-11ee-bb33-fb353e7798cd',
 'a231c0b0-f142-11ee-ba76-fb353e7798cd',
 '72a03d4a-efe9-11ee-b966-fb353e7798cd',
 '96f7a614-f549-11ee-8afa-cb629b0d53e6',
 'c0555ef0-f50f-11ee-8afa-cb629b0d53e6',
 'fcc6fcd2-f013-11ee-b966-fb353e7798cd',
 'd1d090d4-ea7c-11ee-b297-3b0ad9d5d6c6',
 'ba28b352-ec8f-11ee-b297-3b0ad9d5d6c6',
 '3ed4aa16-f1d6-11ee-bae6-fb353e7798cd',
 '868de15e-f3b3-11ee-bb4e-fb353e7798cd',
 'c9c6856c-d33c-11ee-b437-336917683bb8',
 '044d976e-f0e5-11ee-ba20-fb353e7798cd',
 'b76f33be-ea61-11ee-b297-3b0ad9d5d6c6',
 '88dd6fbe-f224-11ee-bb21-fb353e7798cd',
 '8e5c4fc2-f149-11ee-ba7f-fb353e7798cd',
 '837fc882-cb5a-11ee-909c-e1dc60cf66f9',
 'b82476fe-f1f3-11ee-baff-fb353e7798cd',
 '3d2a80f0-ec81-11ee-b297-3b0ad9d5d6c6',
 'aef91c4a-ede5-11ee-9385-ef789ffde1d3',
 '82d39c74-ea59-11ee-b297-3b0ad9d5d6c6',
 'e2079a78-dc1d-11ee-a158-97f8443fd730',
 'dd72fdec-f0cf-11ee-ba0d-fb353e7798cd',
 '04151804-ec20-11ee-b297-3b0ad9d5d6c6',
 '5f6573ba-ed2f-11ee-9385-ef789ffde1d3',
 '80340ab8-d054-11ee-9435-f7e542e2436c',
 'de226278-f25a-11ee-bb4e-fb353e7798cd',
 '5774dcde-f196-11ee-babe-fb353e7798cd',
 'feaf2ba8-d28d-11ee-b437-336917683bb8',
 '8fa6fe80-c869-11ee-a7fc-dd032dba19e8',
 'c8f54ac0-ebd2-11ee-b297-3b0ad9d5d6c6',
 'c2f54552-f06f-11ee-b9a9-fb353e7798cd',
 'f6ac3c82-a445-11ee-88ec-eb6a8d5269b4',
 'f711e68e-f0e1-11ee-ba1f-fb353e7798cd',
 '05c7c824-cab8-11ee-aa4d-1d66adf2f0c7',
 '4d0254fc-ec73-11ee-b297-3b0ad9d5d6c6',
 '04115e66-ea91-11ee-b297-3b0ad9d5d6c6',
 'f9d62032-db2a-11ee-a158-97f8443fd730',
 '8b6a6cfc-ed6d-11ee-9385-ef789ffde1d3',
 'a253145a-d2a6-11ee-b437-336917683bb8',
 'c1b320e2-f079-11ee-b9b0-fb353e7798cd',
 '36663b02-ea87-11ee-b297-3b0ad9d5d6c6',
 '6d2ea45a-c839-11ee-a7fc-dd032dba19e8',
 'c4fca7bc-f18e-11ee-bab8-fb353e7798cd',
 'ce6465b6-f51b-11ee-8afa-cb629b0d53e6',
 'f0bcec4e-ed3e-11ee-9385-ef789ffde1d3',
 '219f7eb8-ef87-11ee-b966-fb353e7798cd',
 '9736e77c-f187-11ee-bab6-fb353e7798cd',
 'c335d84c-a45c-11ee-88ec-eb6a8d5269b4',
 'cf831f42-f353-11ee-bb4e-fb353e7798cd',
 '01e65360-efd4-11ee-b966-fb353e7798cd',
 '99b9f446-f1b2-11ee-bad3-fb353e7798cd',
 'e269948a-ed9d-11ee-9385-ef789ffde1d3',
 '3ec95686-f053-11ee-b988-fb353e7798cd',
 'f9c5e53e-f0ea-11ee-ba28-fb353e7798cd',
 '43a1a35e-f362-11ee-bb4e-fb353e7798cd',
 'd846a080-f115-11ee-ba51-fb353e7798cd',
 'c59a54e0-f179-11ee-baab-fb353e7798cd',
 'd12cd1c4-caec-11ee-909c-e1dc60cf66f9',
 'f570c51c-f15d-11ee-ba91-fb353e7798cd',
 '9798fe24-f143-11ee-ba78-fb353e7798cd',
 '0f3cdf60-f1f6-11ee-bb00-fb353e7798cd',
 'a7c98b32-ebc2-11ee-b297-3b0ad9d5d6c6',
 '71a18322-ecab-11ee-b297-3b0ad9d5d6c6',
 'a6539bd2-cb72-11ee-909c-e1dc60cf66f9',
 '3441fc36-ecca-11ee-b297-3b0ad9d5d6c6',
 '5240e750-ec30-11ee-b297-3b0ad9d5d6c6',
 '1c74d294-f1e4-11ee-baf0-fb353e7798cd',
 '4c88757c-f157-11ee-ba89-fb353e7798cd',
 '8b0593cc-cb4e-11ee-909c-e1dc60cf66f9',
 'df8e3742-ec54-11ee-b297-3b0ad9d5d6c6',
 'c25271be-f3a4-11ee-bb4e-fb353e7798cd',
 '817d6848-efb6-11ee-b966-fb353e7798cd',
 '41b67a28-f52f-11ee-8afa-cb629b0d53e6',
 '84d96f18-f214-11ee-bb13-fb353e7798cd',
 '8dbbbf1c-f0ef-11ee-ba29-fb353e7798cd',
 '64737d98-d312-11ee-b437-336917683bb8',
 'ed352100-cba0-11ee-909c-e1dc60cf66f9',
 '43abeb00-f206-11ee-bb07-fb353e7798cd',
 '96ceec56-f1cf-11ee-bae4-fb353e7798cd',
 'a901fe40-f0fd-11ee-ba39-fb353e7798cd',
 '7a22a34c-f1f0-11ee-bafe-fb353e7798cd',
 '3c415ade-d353-11ee-b437-336917683bb8',
 'c7c02bda-ebe0-11ee-b297-3b0ad9d5d6c6',
 '7f824ea2-f05e-11ee-b993-fb353e7798cd',
 '59c189d8-ed54-11ee-9385-ef789ffde1d3',
 'de493be2-f10f-11ee-ba4b-fb353e7798cd',
 '88a68dd8-eef9-11ee-9385-ef789ffde1d3',
 'be857244-efc0-11ee-b966-fb353e7798cd',
 '64bbe8e0-eb94-11ee-b297-3b0ad9d5d6c6',
 '3a2a78cc-db21-11ee-a158-97f8443fd730',
 '2462c9d0-eecd-11ee-9385-ef789ffde1d3',
 '39ba7438-d0d5-11ee-9435-f7e542e2436c',
 'c4146d46-f074-11ee-b9ac-fb353e7798cd',
 '530de03a-ed79-11ee-9385-ef789ffde1d3',
 '9df14b4e-f172-11ee-baa6-fb353e7798cd',
 '7228e03a-ebf0-11ee-b297-3b0ad9d5d6c6',
 '53fad09e-f0f7-11ee-ba2f-fb353e7798cd',
 '51ef6da6-ca9f-11ee-909c-e1dc60cf66f9',
 '51b74168-f19d-11ee-babf-fb353e7798cd',
 'ba6e1072-9524-11ee-956e-9da2d070324c',
 '2f95c748-f009-11ee-b966-fb353e7798cd',
 'bbbd0cc6-f0dc-11ee-ba1e-fb353e7798cd',
 '7f09f6c6-a5b0-11ee-88ec-eb6a8d5269b4',
 'd24820c8-f197-11ee-babe-fb353e7798cd',
 '47561998-d9c3-11ee-a158-97f8443fd730',
 'ecebb942-f162-11ee-ba97-fb353e7798cd',
 'f12112ba-f1c0-11ee-bada-fb353e7798cd',
 '70060810-eb59-11ee-b297-3b0ad9d5d6c6',
 '19b7ebd0-d9b7-11ee-a158-97f8443fd730',
 'ece2a8be-f047-11ee-b97d-fb353e7798cd',
 '4cf81634-f238-11ee-bb34-fb353e7798cd',
 'b224ef9c-ec10-11ee-b297-3b0ad9d5d6c6',
 'fd1ab258-efa7-11ee-b966-fb353e7798cd',
 'aa5dbcd2-ef10-11ee-9385-ef789ffde1d3',
 'a08a8c7e-f1fb-11ee-bb05-fb353e7798cd',
 'dc39aa14-db32-11ee-a158-97f8443fd730',
 '73bc30cc-f150-11ee-ba84-fb353e7798cd',
 'c9be2042-f0de-11ee-ba1e-fb353e7798cd',
 '60546ef4-edaa-11ee-9385-ef789ffde1d3',
 'c338788a-d324-11ee-b437-336917683bb8',
 '17876fec-ea66-11ee-b297-3b0ad9d5d6c6',
 '853ef120-cad3-11ee-909c-e1dc60cf66f9',
 'cf7148d8-f058-11ee-b98a-fb353e7798cd',
 'f755cf60-f132-11ee-ba6d-fb353e7798cd',
 '58263e34-a45c-11ee-88ec-eb6a8d5269b4',
 'd454c586-f11c-11ee-ba55-fb353e7798cd',
 '1b6aca0e-efdf-11ee-b966-fb353e7798cd',
 'ed7f2038-ea1e-11ee-b297-3b0ad9d5d6c6',
 'f671c05c-a5e4-11ee-88ec-eb6a8d5269b4',
 'd21965e6-f0fa-11ee-ba37-fb353e7798cd',
 '7948628e-f20b-11ee-bb0f-fb353e7798cd',
 '8437f77a-cab7-11ee-909c-e1dc60cf66f9',
 '98692fde-f1a4-11ee-bac6-fb353e7798cd',
 '870cfd32-f1b9-11ee-bad5-fb353e7798cd',
 '85b6e70e-ef7a-11ee-b966-fb353e7798cd',
 '7613801a-edcb-11ee-9385-ef789ffde1d3',
 'c9023e32-ed90-11ee-9385-ef789ffde1d3',
 '961fd9cc-f103-11ee-ba3f-fb353e7798cd',
 'e9a1d768-f23d-11ee-bb39-fb353e7798cd',
 '3344a3c0-f502-11ee-8afa-cb629b0d53e6',
 '5afabc8c-f035-11ee-b966-fb353e7798cd',
 'acd71bc0-ecf4-11ee-9385-ef789ffde1d3',
 '2a61b8a8-f528-11ee-8afa-cb629b0d53e6',
 '154fab12-a43f-11ee-88ec-eb6a8d5269b4',
 '64875cc0-d054-11ee-9435-f7e542e2436c',
 'fa9cba86-f0f0-11ee-ba2a-fb353e7798cd',
 '21376e38-ec01-11ee-b297-3b0ad9d5d6c6',
 'da853e0c-a10f-11ee-981c-d126ddbe9afa',
 'b3ee0dd8-f0d7-11ee-ba18-fb353e7798cd',
 'd1a3a310-f091-11ee-b9ce-fb353e7798cd',
 'de933de8-f112-11ee-ba4d-fb353e7798cd',
 '286e019a-f204-11ee-bb07-fb353e7798cd',
 'dea29156-f123-11ee-ba5d-fb353e7798cd',
 'f0eebb6a-f0dc-11ee-ba1e-fb353e7798cd',
 '25641404-cb66-11ee-909c-e1dc60cf66f9',
 '721a9830-ece6-11ee-b297-3b0ad9d5d6c6',
 '75f83e28-eb77-11ee-b297-3b0ad9d5d6c6',
 '58d78342-f24a-11ee-bb45-fb353e7798cd',
 '3d2d29ec-ef95-11ee-b966-fb353e7798cd',
 '76683d3c-db18-11ee-a158-97f8443fd730',
 '3d8020aa-cb7f-11ee-909c-e1dc60cf66f9',
 'af10e22a-ebb1-11ee-b297-3b0ad9d5d6c6',
 '7fb7b9c0-c881-11ee-a7fc-dd032dba19e8',
 '8c57e8ac-dbec-11ee-a158-97f8443fd730',
 '5fc763f6-f1ab-11ee-bacd-fb353e7798cd',
 'bb4d37d4-f109-11ee-ba46-fb353e7798cd',
 '6d62da08-ec9d-11ee-b297-3b0ad9d5d6c6',
 '68c289fa-dbd4-11ee-a158-97f8443fd730',
 '286c70cc-d2f7-11ee-b437-336917683bb8']

In [20]:
df = cfs.retrieve_gmID_preprocessed_moving_data_v3( gmID = '0f4f0a06-ea98-11ee-b297-3b0ad9d5d6c6', moving_window = 0 )

poi_index = np.where( np.array( df[ 'DisengagementID' ] ) == '0f4f0a06-ea98-11ee-b297-3b0ad9d5d6c6_D1' )[ 0 ][ 0 ]

save_path = f'{ cfs.origin_dir() }/Disengagement_Record/Videos/'

display_poi( df = df, 
             poi_index = poi_index, 
             poi_relative_time_interval = ( -10 * 1e9, 10 * 1e9 ), 
             poi_label = '0f4f0a06-ea98-11ee-b297-3b0ad9d5d6c6_D1', 
             save_path = save_path, 
             animation = True )

In [ ]:
red_df = pd.DataFrame()

green_df = pd.DataFrame()

blue_df = pd.DataFrame()

for i, gmID in enumerate( gmID_list ):

    print( i )

    route_color = cfs.give_route( gmID )

    #

    df = cfs.retrieve_gmID_preprocessed_moving_data_v3( gmID = gmID, moving_window = 0 )

    cfs.solColumns( df )

    #

    indexes = list( np.where( df[ 'solChange' ] == 1 )[ 0 ] )

    for index in indexes:

        row = { 'groupMetadataID' : gmID, 'SolType_Transition' : df[ 'Changesol' ][ index ], 'gmIDInd' : index, 
                'ProgressAlongRoute' : df[ 'ProgressAlongRoute' ][ index ], 'Longitude' : df[ 'longitude' ][ index ], 'Latitude' : df[ 'latitude' ][ index ]}

        row = pd.DataFrame( [ row ] )

        if route_color == 'Red':

            red_df = pd.concat( [ red_df, row ], ignore_index = True )

        elif route_color == 'Green':

            green_df = pd.concat( [ green_df, row ], ignore_index = True )

        elif route_color == 'Blue':

            blue_df = pd.concat( [ blue_df, row ], ignore_index = True )

#

red_df = red_df.sort_values( 'ProgressAlongRoute' )

green_df = green_df.sort_values( 'ProgressAlongRoute' )

blue_df = blue_df.sort_values( 'ProgressAlongRoute' )

#

red_df.to_csv( f'{ cfs.origin_dir() }/SolType_Transition_Record/Locational_Info/Red/Red_SolType_Transitions_v2.csv', index = False )

green_df.to_csv( f'{ cfs.origin_dir() }/SolType_Transition_Record/Locational_Info/Green/Green_SolType_Transitions_v2.csv', index = False )

blue_df.to_csv( f'{ cfs.origin_dir() }/SolType_Transition_Record/Locational_Info/Blue/Blue_SolType_Transitions_v2.csv', index = False )

In [6]:
#gmID_list = gmID_list[ 20 : ]
#gmID_list = gmID_list[ 7 : ]

#gmID_list = gmID_list[ 40 : ]

In [7]:
import time

In [ ]:
for i, gmID in enumerate( gmID_list ):

    df = cfs.retrieve_gmID_preprocessed_moving_data_v3( gmID = gmID, moving_window = 0 )

    #

    indexes = list( np.where( df[ 'BinaryDisengagement' ] == 1 )[ 0 ] )

    DisengagementIDs = df[ 'DisengagementID' ]

    save_path = f'{ cfs.origin_dir() }/Disengagement_Record/Pictures/'

    for j, index in enumerate( indexes ):

        DisengagementID = DisengagementIDs[ index ]

        if not os.path.exists( f'{ save_path }{ DisengagementID }.png' ):

            print( i )

            display_poi( df = df, 
                         poi_index = index, 
                         poi_relative_time_interval = ( -10 * 1e9, 10 * 1e9 ), 
                         poi_label = f'{ DisengagementID }', 
                         save_path = save_path, 
                         animation = False )

        elif os.path.exists( f'{ save_path }{ DisengagementID }.png' ):

            print( f'Skipping { i }' )

    del df

Skipping 0
Skipping 0
Skipping 1
Skipping 1
Skipping 1
Skipping 1
Skipping 1
Skipping 1
Skipping 1
Skipping 2
Skipping 2
Skipping 2
Skipping 2
Skipping 2
Skipping 2
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 3
Skipping 4
Skipping 4
Skipping 4
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 5
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 6
Skipping 7
Skipping 7
Skipping 7
Skipping 7
Skipping 7
Skipping 7
Skipping 8
Skipping 8
Skipping 8
Skipping 8
Skipping 8
Skipping 8
Skipping 8
Skipping 8
Skipping 9
Skipping 9